# Task 3: RNA-Seq Clustering

**Goal:** Cluster cancer samples by gene expression (PCA + UMAP) and overlay HER2 status.

Steps:
1. Load normalized RNA and clinical data
2. Subset to primary tumor samples
3. PCA
4. UMAP
5. Overlay HER2 status and explore other cluster annotations

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

from src.data_utils import load_processed
from src.plotting import plot_scatter_2d, save_fig

## 1. Load data

In [ ]:
rna = load_processed('rna_normalized')
clinical = (
    load_processed('clinical_cleaned')
    .set_index('Patient ID')
    .loc[lambda df: ~df.index.duplicated(keep='first')]  # drop duplicate patient rows
)

print('RNA:', rna.shape)
print('Clinical:', clinical.shape)

## 2. Subset to primary tumor samples

In [ ]:
tumor = rna[rna['sample_type'] == 'Primary Tumor']
gene_cols = [c for c in tumor.columns if c != 'sample_type']
X = tumor[gene_cols]
print('Tumor samples:', X.shape)

## 3. PCA

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=50, random_state=42)
pca_coords = pca.fit_transform(X_scaled)

# Variance explained
plt.figure(figsize=(8, 4))
plt.plot(range(1, 21), pca.explained_variance_ratio_[:20] * 100, 'o-')
plt.xlabel('PC')
plt.ylabel('% Variance Explained')
plt.title('PCA Scree Plot')
save_fig('03_pca_scree')
plt.show()

In [ ]:
# Align HER2 labels to tumor samples
her2_labels = clinical.reindex(tumor.index)['her2_positive'].map({True: 'Positive', False: 'Negative'}).fillna('Unknown')

pca_df = pd.DataFrame(pca_coords[:, :2], index=tumor.index, columns=['PC1', 'PC2'])
plot_scatter_2d(pca_df, her2_labels, title='PCA — colored by HER2 status', x_label='PC1', y_label='PC2')
save_fig('03_pca_her2')
plt.show()

In [ ]:
# PC1 vs PC3 colored by HER2 status
pc1_pc3 = pd.DataFrame(pca_coords[:, [0, 2]], index=tumor.index, columns=['PC1', 'PC3'])
plot_scatter_2d(pc1_pc3, her2_labels, title='PCA — PC1 vs PC3 colored by HER2 status', x_label='PC1', y_label='PC3')
save_fig('03_pca_pc1_pc3_her2')
plt.show()

## 4. UMAP

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
umap_coords = reducer.fit_transform(pca_coords[:, :30])  # Use top 30 PCs as input

umap_df = pd.DataFrame(umap_coords, index=tumor.index, columns=['UMAP1', 'UMAP2'])
plot_scatter_2d(umap_df, her2_labels, title='UMAP — colored by HER2 status', x_label='UMAP1', y_label='UMAP2')
save_fig('03_umap_her2')
plt.show()

## 5. Explore other cluster annotations

TODO: overlay stage, PAM50 subtype, ER/PR status, etc.

In [ ]:
# Example: ERBB2 expression on UMAP
erbb2_expr = tumor['ERBB2'] if 'ERBB2' in tumor.columns else None
if erbb2_expr is not None:
    plt.figure(figsize=(7, 5))
    sc = plt.scatter(umap_df['UMAP1'], umap_df['UMAP2'], c=erbb2_expr, cmap='RdYlBu_r', s=15, alpha=0.7)
    plt.colorbar(sc, label='ERBB2 log2 CPM')
    plt.title('UMAP — ERBB2 expression')
    save_fig('03_umap_erbb2_expr')
    plt.show()